# Task 04 – Marketing Campaign Exploratory Data Analysis
## SkillCraft Technology Internship

**Objective:** Analyze customer demographics, product spending, shopping channels and campaign response using the supplied marketing campaign dataset.

**Data note:** This dataset contains customer purchase amounts and campaign response indicators, but it does not contain channel-level advertising spend or revenue attribution. Therefore, true ROI and channel profitability cannot be calculated.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("marketing_campaign.csv", sep="\\t")
df.columns = df.columns.str.replace("\\ufeff", "", regex=False).str.strip()
df.head()


## 1. Inspect dataset and data quality

In [ ]:
print("Shape:", df.shape)
print(df.info())
print("Missing values:\\n", df.isna().sum()[df.isna().sum() > 0])
print("Duplicate customer IDs:", df["ID"].duplicated().sum())


## 2. Clean and prepare data

In [ ]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], errors="coerce")
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Fill missing Income with the median and retain a missing-value flag
df["Income_Missing_Flag"] = df["Income"].isna().astype(int)
df["Income"] = df["Income"].fillna(df["Income"].median())

df = df.drop_duplicates(subset=["ID"]).copy()

product_cols = ["MntWines","MntFruits","MntMeatProducts",
                "MntFishProducts","MntSweetProducts","MntGoldProds"]
df["Total_Spend"] = df[product_cols].sum(axis=1)
df["Total_Purchases"] = df[["NumWebPurchases","NumCatalogPurchases",
                            "NumStorePurchases"]].sum(axis=1)
df["Campaigns_Accepted"] = df[["AcceptedCmp1","AcceptedCmp2",
                               "AcceptedCmp3","AcceptedCmp4","AcceptedCmp5"]].sum(axis=1)
df.head()


## 3. Key performance indicators

In [ ]:
customer_count = len(df)
response_count = int(df["Response"].sum())
response_rate = df["Response"].mean()
average_spend = df["Total_Spend"].mean()
total_spend = df["Total_Spend"].sum()

print(f"Customers: {customer_count:,}")
print(f"Responders: {response_count:,}")
print(f"Response rate: {response_rate:.2%}")
print(f"Average spend per customer: {average_spend:,.2f}")
print(f"Total recorded spend: {total_spend:,.2f}")


## 4. Product category and shopping channel analysis

In [ ]:
product_cols = ["MntWines","MntFruits","MntMeatProducts",
                "MntFishProducts","MntSweetProducts","MntGoldProds"]
df[product_cols].sum().sort_values().plot(kind="barh", title="Spend by Product Category")
plt.xlabel("Recorded spend")
plt.show()

channel_cols = ["NumWebPurchases","NumCatalogPurchases","NumStorePurchases"]
df[channel_cols].sum().sort_values().plot(kind="barh", title="Purchases by Channel")
plt.xlabel("Number of purchases")
plt.show()


## 5. Customer segment analysis

In [ ]:
segment = df.groupby("Education").agg(
    Customers=("ID","count"),
    Average_Spend=("Total_Spend","mean"),
    Response_Rate=("Response","mean")
).sort_values("Customers", ascending=False)
display(segment)

segment["Average_Spend"].sort_values().plot(kind="barh", title="Average Spend by Education")
plt.xlabel("Average spend")
plt.show()


## 6. Campaign response analysis

In [ ]:
response_rate_by_marital = df.groupby("Marital_Status")["Response"].mean().sort_values()
display(response_rate_by_marital)

response_rate_by_marital.plot(kind="barh", title="Response Rate by Marital Status")
plt.xlabel("Response rate")
plt.show()


## 7. Business insights and limitations

- Product spending totals identify categories with the largest recorded customer expenditure.
- Purchase counts by web, catalog and store describe channel usage; they do not measure channel revenue or marketing ROI.
- Response rate is calculated as customers with `Response = 1` divided by all customers.
- Segment-level comparisons are descriptive and should not be interpreted as causal effects.
- The dataset does not provide advertising cost by channel, campaign-level spend, or attributable channel revenue. ROI cannot be calculated.
- Income was median-imputed for analysis, with a separate flag retained to identify originally missing values.


## 8. Export cleaned dataset

In [ ]:
df.to_csv("Task_04_Cleaned_Marketing_Campaign.csv", index=False)
print("Cleaned dataset exported.")
